# Projeto Integrador â€” AnÃ¡lise Quantitativa de uma Carteira

Este notebook integra os seis capÃ­tulos do curso em uma anÃ¡lise reprodutÃ­vel de uma carteira brasileira. Os tickers de referÃªncia sÃ£o PETR4, VALE3, ITUB4 e BOVA11.

O notebook separa rigorosamente **IN SAMPLE** e **OUT OF SAMPLE**. ParÃ¢metros de retorno esperado, covariÃ¢ncia, PCA, beta e pesos sÃ£o estimados somente no perÃ­odo in sample. O perÃ­odo out of sample serve apenas para avaliaÃ§Ã£o.

Quando nÃ£o hÃ¡ dados externos, usamos um provedor local determinÃ­stico. Isso mantÃ©m o projeto executÃ¡vel em testes, Binder e ambientes sem internet.

## Roteiro de leitura do projeto

Leia o projeto como uma investigação, não como uma sequência de comandos: **pergunta**, **dados**, **hipóteses**, **estimação in sample**, **decisão**, **avaliação out of sample**, **cenários**, **interpretação** e **limitações**. Em cada tabela, escreva mentalmente uma conclusão e uma razão pela qual ela pode estar errada.

## 1. Ambiente e DataProvider

O contrato `DataProvider` desacopla a origem dos preÃ§os da anÃ¡lise. Um provedor externo pode ser conectado depois sem mudar as cÃ©lulas de modelagem. O provedor local abaixo simula preÃ§os com fatores comuns e choques idiossincrÃ¡ticos, usando os nomes de ativos brasileiros como interface didÃ¡tica.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.decomposition import PCA

from quantfinance.linear_algebra import pca_from_covariance
from quantfinance.performance import (
    omega_ratio,
    sharpe_ratio,
    sortino_ratio,
)
from quantfinance.portfolio import (
    efficient_frontier,
    global_minimum_variance,
    portfolio_sharpe,
    tangency_portfolio,
)
from quantfinance.returns import log_return, simple_return

np.set_printoptions(precision=5, suppress=True)
pd.set_option("display.float_format", lambda value: f"{value:,.5f}")

In [ ]:
class DataProvider(Protocol):
    def load_prices(self, tickers: list[str], periods: int = 1_000) -> pd.DataFrame:
        ...


@dataclass
class LocalExampleProvider:
    seed: int = 2026

    def load_prices(self, tickers: list[str], periods: int = 1_000) -> pd.DataFrame:
        rng = np.random.default_rng(self.seed)
        market = rng.normal(0.00035, 0.012, periods)
        sector = rng.normal(0.00010, 0.007, periods)
        betas = np.array([1.25, 1.05, 0.85, 1.00])
        sector_loadings = np.array([0.30, 0.45, 0.20, 0.10])
        idiosyncratic = rng.normal(0.0, [0.018, 0.016, 0.010, 0.006], (periods, len(tickers)))
        returns = (
            market[:, None] * betas
            + sector[:, None] * sector_loadings
            + idiosyncratic
        )
        dates = pd.bdate_range("2018-01-01", periods=periods)
        prices = 100.0 * np.exp(np.cumsum(returns, axis=0))
        return pd.DataFrame(prices, index=dates, columns=tickers)


class CsvDataProvider:
    def __init__(self, path: str | Path, date_column: str = "Date") -> None:
        self.path = Path(path)
        self.date_column = date_column

    def load_prices(self, tickers: list[str], periods: int = 1_000) -> pd.DataFrame:
        prices = pd.read_csv(self.path, parse_dates=[self.date_column]).set_index(self.date_column)
        return prices.loc[:, tickers].dropna().tail(periods)


tickers = ["PETR4", "VALE3", "ITUB4", "BOVA11"]
provider: DataProvider = LocalExampleProvider(seed=2026)
prices = provider.load_prices(tickers, periods=1_000)
prices.head()

## 2. Limpeza, preÃ§os e retornos

PreÃ§os sÃ£o ordenados no tempo, duplicatas sÃ£o removidas e observaÃ§Ãµes incompletas sÃ£o descartadas. O retorno simples Ã© adequado para comunicar desempenho; o log-retorno Ã© aditivo no tempo.

In [ ]:
prices = prices.sort_index().loc[~prices.index.duplicated()].dropna(how="all").ffill().dropna()
simple_returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
prices.plot(ax=axes[0], title="PreÃ§os simulados")
(1 + simple_returns).cumprod().plot(ax=axes[1], title="Crescimento acumulado")
plt.tight_layout()
plt.show()

## 3. SeparaÃ§Ã£o temporal: IN SAMPLE e OUT OF SAMPLE

O corte temporal ocorre antes de qualquer estimaÃ§Ã£o. Os parÃ¢metros abaixo serÃ£o calculados somente no in sample. Nenhuma observaÃ§Ã£o out of sample participa da escolha dos pesos ou da calibraÃ§Ã£o.

In [ ]:
split = int(len(simple_returns) * 0.70)
in_sample = simple_returns.iloc[:split].copy()
out_of_sample = simple_returns.iloc[split:].copy()
print("IN SAMPLE:", in_sample.index.min().date(), "atÃ©", in_sample.index.max().date(), len(in_sample), "observaÃ§Ãµes")
print("OUT OF SAMPLE:", out_of_sample.index.min().date(), "atÃ©", out_of_sample.index.max().date(), len(out_of_sample), "observaÃ§Ãµes")

## 4. EstatÃ­sticas e Normal versus Student-t â€” IN SAMPLE

A Normal Ã© uma referÃªncia Ãºtil, mas pode subestimar caudas. Ajustamos Normal e Student-t apenas no in sample e comparamos quantis extremos, skewness e excesso de kurtosis.

In [ ]:
descriptive = in_sample.agg(["mean", "std", "skew", "kurt"]).T
descriptive["q01"] = in_sample.quantile(0.01)
descriptive["q99"] = in_sample.quantile(0.99)
display(descriptive)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for axis, ticker in zip(axes.flat, tickers):
    values = in_sample[ticker]
    mean, std = values.mean(), values.std(ddof=1)
    degrees, location, scale = stats.t.fit(values)
    grid = np.linspace(values.quantile(0.001), values.quantile(0.999), 300)
    axis.hist(values, bins=45, density=True, alpha=0.55, label="amostra")
    axis.plot(grid, stats.norm.pdf(grid, mean, std), label="Normal")
    axis.plot(grid, stats.t.pdf(grid, degrees, loc=location, scale=scale), label="Student-t")
    axis.set_title(ticker)
    axis.legend()
plt.tight_layout()
plt.show()

## 5. DependÃªncia e PCA â€” IN SAMPLE

CorrelaÃ§Ã£o e covariÃ¢ncia sÃ£o estimadas somente no perÃ­odo de treinamento. A PCA resume as direÃ§Ãµes de maior variÃ¢ncia; os loadings nÃ£o sÃ£o previsÃµes e podem mudar quando o regime muda.

In [ ]:
expected_returns = in_sample.mean().to_numpy() * 252
covariance = in_sample.cov().to_numpy() * 252
correlation = in_sample.corr()
display(correlation)
display(pd.DataFrame(covariance, index=tickers, columns=tickers))

pca_manual = pca_from_covariance(covariance)
pca_library = PCA().fit(in_sample)
print("VariÃ¢ncia explicada manual:", pca_manual.explained_variance_ratio)
print("VariÃ¢ncia explicada sklearn:", pca_library.explained_variance_ratio_)
print("Loadings dos dois primeiros fatores:")
display(pd.DataFrame(pca_manual.loadings[:, :2], index=tickers, columns=["PC1", "PC2"]))
plt.figure(figsize=(7, 4))
plt.bar(np.arange(1, len(tickers) + 1), pca_manual.explained_variance_ratio)
plt.xlabel("Componente")
plt.ylabel("VariÃ¢ncia explicada")
plt.title("PCA da covariÃ¢ncia IN SAMPLE")
plt.show()

## 6. Beta CAPM e regressÃµes â€” IN SAMPLE

BOVA11 serÃ¡ usado como proxy de mercado. Para cada ativo, estimamos:

$$R_i-R_f=\alpha+\beta(R_m-R_f)+\varepsilon.$$

Os betas e fatores desta seÃ§Ã£o nÃ£o usam dados out of sample.

In [ ]:
risk_free_daily = 0.0001
market = in_sample["BOVA11"]
capm_rows = []
for ticker in tickers:
    model = sm.OLS(
        in_sample[ticker] - risk_free_daily,
        sm.add_constant(market - risk_free_daily),
    ).fit()
    capm_rows.append({
        "ticker": ticker,
        "alpha": model.params.iloc[0],
        "beta": model.params.iloc[1],
        "r_squared": model.rsquared,
        "residual_risk": model.resid.std(ddof=1),
    })
capm_table = pd.DataFrame(capm_rows).set_index("ticker")
display(capm_table)

## 7. Carteiras: equal weight, GMV, Markowitz e tangÃªncia â€” IN SAMPLE

Os pesos sÃ£o escolhidos usando apenas mÃ©dias e covariÃ¢ncias in sample. A fronteira Ã© uma coleÃ§Ã£o de carteiras de variÃ¢ncia mÃ­nima para retornos-alvo.

In [ ]:
equal_weights = np.repeat(1 / len(tickers), len(tickers))
gmv_weights = global_minimum_variance(covariance)
tangency_weights = tangency_portfolio(expected_returns, covariance, risk_free_rate=0.02)
target_grid = np.linspace(expected_returns.min(), expected_returns.max(), 30)
frontier = efficient_frontier(expected_returns, covariance, target_grid)

portfolio_table = pd.DataFrame({
    "Equal weight": equal_weights,
    "GMV": gmv_weights,
    "Tangency": tangency_weights,
}, index=tickers)
display(portfolio_table)
for name, weights in [("Equal weight", equal_weights), ("GMV", gmv_weights), ("Tangency", tangency_weights)]:
    print(name, "retorno anual:", weights @ expected_returns, "vol:", np.sqrt(weights @ covariance @ weights), "Sharpe:", portfolio_sharpe(weights, expected_returns, covariance, 0.02))

plt.figure(figsize=(8, 5))
plt.plot(frontier.volatilities, frontier.target_returns, label="fronteira")
plt.scatter([np.sqrt(gmv_weights @ covariance @ gmv_weights)], [gmv_weights @ expected_returns], label="GMV")
plt.scatter([np.sqrt(tangency_weights @ covariance @ tangency_weights)], [tangency_weights @ expected_returns], label="TangÃªncia")
plt.xlabel("Volatilidade anualizada")
plt.ylabel("Retorno esperado anualizado")
plt.legend()
plt.title("Markowitz IN SAMPLE")
plt.show()

## 8. AvaliaÃ§Ã£o OUT OF SAMPLE

Os pesos abaixo foram congelados no fim do in sample. O out of sample nÃ£o recalibra mÃ©dias, covariÃ¢ncias, PCA, betas ou pesos. Isso evita data leakage.

In [ ]:
def portfolio_series(weights: np.ndarray, returns: pd.DataFrame) -> pd.Series:
    return returns @ weights

portfolio_returns_out = pd.DataFrame({
    "Equal weight": portfolio_series(equal_weights, out_of_sample),
    "GMV": portfolio_series(gmv_weights, out_of_sample),
    "Tangency": portfolio_series(tangency_weights, out_of_sample),
})

out_metrics = pd.DataFrame({
    name: {
        "retorno acumulado": (1 + values).prod() - 1,
        "volatilidade anual": values.std(ddof=1) * np.sqrt(252),
        "Sharpe": sharpe_ratio(values.to_numpy(), risk_free_rate=risk_free_daily, periods=252),
        "Sortino": sortino_ratio(values.to_numpy(), threshold=risk_free_daily, periods=252),
        "Omega": omega_ratio(values.to_numpy(), threshold=risk_free_daily),
    }
    for name, values in portfolio_returns_out.items()
}).T
display(out_metrics)
(1 + portfolio_returns_out).cumprod().plot(figsize=(10, 4), title="Avaliação OUT OF SAMPLE")
plt.show()

## 9. Monte Carlo da carteira e cenÃ¡rios

A simulaÃ§Ã£o usa mÃ©dias e covariÃ¢ncias estimadas apenas no in sample. Ela Ã© um exercÃ­cio de modelo, nÃ£o uma previsÃ£o. Os cenÃ¡rios abaixo alteram choques e correlaÃ§Ãµes para avaliar sensibilidade.

In [ ]:
simulation_count = 5_000
horizon_days = 252
rng = np.random.default_rng(99)
simulated_asset_returns = rng.multivariate_normal(
    expected_returns / 252,
    covariance / 252,
    size=(simulation_count, horizon_days),
)
simulated_portfolio_returns = simulated_asset_returns @ tangency_weights
simulated_terminal_wealth = np.prod(1.0 + simulated_portfolio_returns, axis=1)
print("Monte Carlo média riqueza relativa:", simulated_terminal_wealth.mean())
print("Monte Carlo p05/p95:", np.quantile(simulated_terminal_wealth, [0.05, 0.95]))

stress_returns = expected_returns - 2 * np.sqrt(np.diag(covariance))
stress_return = tangency_weights @ stress_returns
correlation_stress = np.full_like(correlation, 0.75)
np.fill_diagonal(correlation_stress, 1.0)
stress_volatilities = np.sqrt(np.diag(covariance))
stress_covariance = np.diag(stress_volatilities) @ correlation_stress @ np.diag(stress_volatilities)
print("Cenário -2 desvios por ativo:", stress_return)
print("Volatilidade tangência base:", np.sqrt(tangency_weights @ covariance @ tangency_weights))
print("Volatilidade com correlação estressada:", np.sqrt(tangency_weights @ stress_covariance @ tangency_weights))

plt.figure(figsize=(8, 4))
plt.hist(simulated_terminal_wealth, bins=60, density=True)
plt.axvline(np.quantile(simulated_terminal_wealth, 0.05), color="red", linestyle="--", label="p05")
plt.title("Distribuição Monte Carlo da riqueza relativa")
plt.legend()
plt.show()

## O que este modelo nÃ£o sabe

Este projeto Ã© uma anÃ¡lise quantitativa educacional. Ele nÃ£o sabe:

- que parÃ¢metros sÃ£o instÃ¡veis ao longo do tempo;
- estimar expected returns com precisÃ£o confiÃ¡vel;
- que correlaÃ§Ãµes mudam em crises;
- representar completamente nÃ£o-normalidade e caudas;
- identificar regimes futuros;
- custos de transaÃ§Ã£o;
- liquidez;
- slippage;
- survivorship bias;
- overfitting e data snooping;
- mudanÃ§as de composiÃ§Ã£o dos Ã­ndices;
- restriÃ§Ãµes operacionais, tributÃ¡rias ou de short selling.

Um bom resultado in sample nÃ£o Ã© evidÃªncia suficiente de uma estratÃ©gia negociÃ¡vel. A separaÃ§Ã£o temporal, a validaÃ§Ã£o fora da amostra e a anÃ¡lise de sensibilidade reduzem, mas nÃ£o eliminam, esses riscos.

## Perguntas finais para o estudante

1. Qual decisão foi tomada somente com dados `IN SAMPLE`?
2. Qual resultado mudou mais `OUT OF SAMPLE`?
3. Qual hipótese explica essa mudança?
4. Que teste adicional reduziria o risco de overfitting?
5. Que custo ou restrição operacional ainda não entrou no modelo?

Uma análise quantitativa madura termina com perguntas verificáveis, não com um ranking isolado de carteiras.